# 00 - Limpieza y preparacion del dataset de trabajo

Este notebook documenta la construccion del dataset analitico usado en los cuadernos posteriores. El objetivo no es repetir todo el procesamiento pesado desde los archivos originales, sino dejar una auditoria reproducible de las decisiones principales: integracion de fuentes, eliminacion de nulos, correccion de valores imposibles, creacion de variables derivadas y tratamiento de valores atipicos.

La unidad de analisis es una lista completa de eBird en Bogota D.C. vinculada con las condiciones ambientales y de calidad del aire medidas por la estacion RMCAB mas cercana en la misma hora de observacion.

In [1]:
from pathlib import Path
import json
import os

import numpy as np
import pandas as pd

from IPython.display import display

# Permite ejecutar el notebook desde la raiz del proyecto o desde Notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == "Notebooks" else Path.cwd()
os.chdir(ROOT)

DATA_DIR = ROOT / "Data"
PROCESSED_DIR = DATA_DIR / "Processed"
RAW_DIR = DATA_DIR / "Raw"
OUTPUTS_DIR = ROOT / "Outputs"

with open(OUTPUTS_DIR / "analysis_preparation_summary.json", encoding="utf-8") as fh:
    prep_summary = json.load(fh)

master = pd.read_parquet(PROCESSED_DIR / "master_dataset.parquet")
analysis = pd.read_parquet(PROCESSED_DIR / "analysis_dataset.parquet")
env_complete = pd.read_parquet(PROCESSED_DIR / "analysis_env_complete.parquet")
reg_complete = pd.read_parquet(PROCESSED_DIR / "analysis_regression_complete.parquet")
env_reduced = pd.read_parquet(PROCESSED_DIR / "analysis_env_reduced_no_radiation.parquet")
reg_reduced = pd.read_parquet(PROCESSED_DIR / "analysis_regression_reduced_no_radiation.parquet")
ebird_shannon = pd.read_parquet(PROCESSED_DIR / "ebird_shannon.parquet")
rmcab = pd.read_parquet(PROCESSED_DIR / "rmcab_merged.parquet")

print(f"Raiz del proyecto: {ROOT}")
print(f"Dataset maestro: {master.shape[0]:,} filas x {master.shape[1]} columnas")
print(f"Dataset analitico: {analysis.shape[0]:,} filas x {analysis.shape[1]} columnas")

Raiz del proyecto: c:\Users\Miguel Camargo\Desktop\Parcial FInal Estadistica
Dataset maestro: 33,323 filas x 22 columnas
Dataset analitico: 33,308 filas x 80 columnas


## 1. Fuentes integradas

El dataset de trabajo se construyo a partir de tres bloques de informacion:

- **eBird:** listas completas de observacion de aves en Bogota, usadas para calcular el indice de diversidad de Shannon por evento de muestreo.
- **RMCAB:** contaminantes y variables meteorologicas horarias por estacion de monitoreo.
- **IDEAM/RMCAB precipitacion:** registros horarios usados para representar lluvia; en el dataset final se trabaja como `Precipitacion` y como indicador binario `Lluvia`.

Los archivos originales permanecen en `Data/Raw/` y los productos intermedios/finales en `Data/Processed/`.

In [2]:
raw_inventory = pd.DataFrame({
    "bloque": ["eBird", "RMCAB contaminacion", "RMCAB variables ambientales", "IDEAM precipitacion"],
    "ruta": [
        "Data/Raw/ebird",
        "Data/Raw/contaminacion",
        "Data/Raw/variables_ambientales",
        "Data/Raw/precipitacion_ideam_api",
    ],
})
raw_inventory["existe"] = raw_inventory["ruta"].map(lambda p: (ROOT / p).exists())
raw_inventory["archivos_detectados"] = raw_inventory["ruta"].map(
    lambda p: sum(1 for _ in (ROOT / p).rglob("*") if _.is_file()) if (ROOT / p).exists() else 0
)
display(raw_inventory)

processed_inventory = pd.DataFrame([
    {"archivo": "ebird_shannon.parquet", "descripcion": "Listas eBird filtradas con indice de Shannon", "filas": len(ebird_shannon), "columnas": ebird_shannon.shape[1]},
    {"archivo": "rmcab_merged.parquet", "descripcion": "Mediciones horarias RMCAB en formato largo", "filas": len(rmcab), "columnas": rmcab.shape[1]},
    {"archivo": "master_dataset.parquet", "descripcion": "Cruce espacio-temporal eBird + RMCAB", "filas": len(master), "columnas": master.shape[1]},
    {"archivo": "analysis_dataset.parquet", "descripcion": "Dataset con limpieza basica y variables derivadas", "filas": len(analysis), "columnas": analysis.shape[1]},
    {"archivo": "analysis_env_complete.parquet", "descripcion": "Casos completos para analisis ambiental con 10 variables", "filas": len(env_complete), "columnas": env_complete.shape[1]},
    {"archivo": "analysis_regression_complete.parquet", "descripcion": "Casos completos para regresion con ambiente + esfuerzo", "filas": len(reg_complete), "columnas": reg_complete.shape[1]},
])
display(processed_inventory)

,bloque,ruta,existe,archivos_detectados
0,eBird,Data/Raw/ebird,True,10
1,RMCAB contaminacion,Data/Raw/contaminacion,True,24
2,RMCAB variables ambientales,Data/Raw/variables_ambientales,True,20
3,IDEAM precipitacion,Data/Raw/precipitacion_ideam_api,True,37


,archivo,descripcion,filas,columnas
0,ebird_shannon.parquet,Listas eBird filtradas con indice de Shannon,38609,11
1,rmcab_merged.parquet,Mediciones horarias RMCAB en formato largo,913358,12
2,master_dataset.parquet,Cruce espacio-temporal eBird + RMCAB,33323,22
3,analysis_dataset.parquet,Dataset con limpieza basica y variables derivadas,33308,80
4,analysis_env_complete.parquet,Casos completos para analisis ambiental con 10...,6178,80
5,analysis_regression_complete.parquet,Casos completos para regresion con ambiente + ...,5298,80


## 2. Construccion del dataset maestro

El flujo de construccion fue el siguiente:

1. Se filtraron las listas de eBird para conservar eventos completos y observaciones con conteos numericos suficientes para calcular Shannon.
2. Se agregaron las abundancias por lista y se calculo `Shannon_Index` como variable respuesta ecologica.
3. Los archivos horarios de RMCAB se pasaron de formato ancho a formato largo, usando una fila por `Fecha_Hora` y `Estacion`.
4. Para cada lista eBird se identifico la estacion RMCAB mas cercana mediante distancia geografica. Se conservaron listas dentro de un radio maximo de 5 km.
5. Se cruzo cada lista con la medicion ambiental de la estacion mas cercana en la hora de inicio de observacion.

In [3]:
pipeline_rows = pd.DataFrame([
    {"etapa": "Listas eBird con Shannon", "filas": len(ebird_shannon)},
    {"etapa": "Listas cruzadas con estacion RMCAB <= 5 km", "filas": len(master)},
    {"etapa": "Dataset analitico tras limpieza basica", "filas": len(analysis)},
    {"etapa": "Casos completos: ambiente 10 variables", "filas": len(env_complete)},
    {"etapa": "Casos completos: regresion ambiente + esfuerzo", "filas": len(reg_complete)},
    {"etapa": "Casos completos sin radiacion solar", "filas": len(env_reduced)},
])
pipeline_rows["retencion_vs_master_pct"] = (pipeline_rows["filas"] / len(master) * 100).round(2)
pipeline_rows.loc[0, "retencion_vs_master_pct"] = np.nan
display(pipeline_rows)

coverage = pd.Series({
    "fecha_minima": master["Fecha_Hora"].min(),
    "fecha_maxima": master["Fecha_Hora"].max(),
    "estaciones_rmcab": master["Estacion"].nunique(),
    "distancia_media_km": master["Distance_to_Station_km"].mean(),
    "distancia_maxima_km": master["Distance_to_Station_km"].max(),
    "eventos_duplicados": master["SAMPLING EVENT IDENTIFIER"].duplicated().sum(),
})
display(coverage.to_frame("valor"))

,etapa,filas,retencion_vs_master_pct
0,Listas eBird con Shannon,38609,NaN
1,Listas cruzadas con estacion RMCAB <= 5 km,33323,100.00
2,Dataset analitico tras limpieza basica,33308,99.95
3,Casos completos: ambiente 10 variables,6178,18.54
4,Casos completos: regresion ambiente + esfuerzo,5298,15.90
5,Casos completos sin radiacion solar,9061,27.19


,valor
fecha_minima,2020-01-01 07:00:00
fecha_maxima,2025-12-31 14:00:00
estaciones_rmcab,19
distancia_media_km,2.016904
distancia_maxima_km,4.989844
eventos_duplicados,0


## 3. Revision de nulos y criterio de eliminacion

Los valores faltantes provienen principalmente de interrupciones de sensores, estaciones que no reportan todas las variables y horas sin medicion disponible. En este proyecto se opto por **eliminacion por casos completos** en los datasets finales de modelamiento, porque:

- Las tecnicas multivariadas usadas despues requieren matrices comparables entre observaciones.
- Imputar contaminantes o clima a nivel horario podria introducir una senal artificial, especialmente cuando los faltantes dependen de estacion, sensor u hora.
- Se conservaron versiones alternativas sin `Radiacion_Solar`, pues esta variable es la que mas reduce el tamano muestral.

In [4]:
base_vars = [
    "PM2.5", "PM10", "NO2", "CO", "O3", "Temperatura", "Humedad", "Viento",
    "Precipitacion", "Radiacion_Solar", "DURATION MINUTES", "EFFORT DISTANCE KM", "Shannon_Index"
]

missing = pd.DataFrame({
    "nulos_master": master[base_vars].isna().sum(),
    "pct_master": (master[base_vars].isna().mean() * 100).round(2),
    "nulos_analysis": analysis[base_vars].isna().sum(),
    "pct_analysis": (analysis[base_vars].isna().mean() * 100).round(2),
}).sort_values("pct_master", ascending=False)
display(missing)

case_sets = pd.DataFrame([
    {"dataset": "master_dataset", "criterio": "Cruce espacio-temporal inicial", "filas": len(master)},
    {"dataset": "analysis_dataset", "criterio": "Limpieza basica; conserva nulos para auditoria", "filas": len(analysis)},
    {"dataset": "analysis_env_complete", "criterio": "dropna en 10 variables ambientales", "filas": len(env_complete)},
    {"dataset": "analysis_regression_complete", "criterio": "dropna en ambiente + duracion + distancia", "filas": len(reg_complete)},
    {"dataset": "analysis_env_reduced_no_radiation", "criterio": "dropna ambiental excluyendo Radiacion_Solar", "filas": len(env_reduced)},
    {"dataset": "analysis_regression_reduced_no_radiation", "criterio": "dropna regresion excluyendo Radiacion_Solar", "filas": len(reg_reduced)},
])
case_sets["retencion_vs_master_pct"] = (case_sets["filas"] / len(master) * 100).round(2)
display(case_sets)

,nulos_master,pct_master,nulos_analysis,pct_analysis
Radiacion_Solar,18554,55.68,18539,55.66
Humedad,12812,38.45,12797,38.42
O3,10251,30.76,10236,30.73
EFFORT DISTANCE KM,9021,27.07,9006,27.04
NO2,8906,26.73,8891,26.69
Viento,8543,25.64,8528,25.60
CO,8097,24.30,8089,24.29
Temperatura,7300,21.91,7285,21.87
PM10,4937,14.82,4922,14.78
PM2.5,3909,11.73,3894,11.69


,dataset,criterio,filas,retencion_vs_master_pct
0,master_dataset,Cruce espacio-temporal inicial,33323,100.00
1,analysis_dataset,Limpieza basica; conserva nulos para auditoria,33308,99.95
2,analysis_env_complete,dropna en 10 variables ambientales,6178,18.54
3,analysis_regression_complete,dropna en ambiente + duracion + distancia,5298,15.90
4,analysis_env_reduced_no_radiation,dropna ambiental excluyendo Radiacion_Solar,9061,27.19
5,analysis_regression_reduced_no_radiation,dropna regresion excluyendo Radiacion_Solar,7276,21.83


## 4. Valores imposibles y transformaciones

Antes de estandarizar, se revisaron variables que por definicion no deberian ser negativas: contaminantes, precipitacion, radiacion, velocidad del viento y variables de esfuerzo. Los pocos valores negativos detectados en `CO` se trataron como mediciones invalidas y se pasaron a `NaN`; posteriormente quedaron excluidos en los datasets de casos completos.

Tambien se crearon variables auxiliares para el analisis:

- `Anio`, `Mes` y `Hora` para controlar estructura temporal.
- `Lluvia`, indicador binario derivado de `Precipitacion > 0`.
- Transformaciones `log1p_*` para variables asimetricas no negativas.
- Variables estandarizadas `z_env_*` y `z_reg_*` para comparar magnitudes en analisis multivariado y regresion.
- `Diversidad_Grupo`, que separa baja diversidad, rango medio y alta diversidad usando cuartiles de Shannon.

In [5]:
non_negative_vars = [
    "PM2.5", "PM10", "NO2", "CO", "O3", "Viento", "Precipitacion", "Radiacion_Solar",
    "DURATION MINUTES", "EFFORT DISTANCE KM"
]

negative_audit = pd.DataFrame({
    "variable": non_negative_vars,
    "negativos_detectados": [int((master[v] < 0).sum()) for v in non_negative_vars],
    "negativos_convertidos_a_nan": [prep_summary["negative_values_set_to_nan"].get(v, 0) for v in non_negative_vars],
})
display(negative_audit)

derived_cols = [c for c in analysis.columns if c.startswith("log1p_") or c.startswith("z_env_") or c.startswith("z_reg_")]
print(f"Variables derivadas log/z detectadas: {len(derived_cols)}")
print(derived_cols)

print("\nCuartiles usados para grupos de Shannon:")
print(f"Q1 = {prep_summary['shannon_q1']:.3f}")
print(f"Q3 = {prep_summary['shannon_q3']:.3f}")
display(analysis["Diversidad_Grupo"].value_counts().rename_axis("grupo").reset_index(name="n"))

,variable,negativos_detectados,negativos_convertidos_a_nan
0,PM2.5,0,0
1,PM10,0,0
2,NO2,0,0
3,CO,7,7
4,O3,0,0
5,Viento,0,0
6,Precipitacion,0,0
7,Radiacion_Solar,0,0
8,DURATION MINUTES,0,0
9,EFFORT DISTANCE KM,0,0


Variables derivadas log/z detectadas: 31
['log1p_PM2_5', 'log1p_PM10', 'log1p_NO2', 'log1p_CO', 'log1p_O3', 'log1p_Precipitacion', 'log1p_Radiacion_Solar', 'log1p_DURATION_MINUTES', 'log1p_EFFORT_DISTANCE_KM', 'z_env_PM2.5', 'z_env_PM10', 'z_env_NO2', 'z_env_CO', 'z_env_O3', 'z_env_Temperatura', 'z_env_Humedad', 'z_env_Viento', 'z_env_Lluvia', 'z_env_Radiacion_Solar', 'z_reg_PM2.5', 'z_reg_PM10', 'z_reg_NO2', 'z_reg_CO', 'z_reg_O3', 'z_reg_Temperatura', 'z_reg_Humedad', 'z_reg_Viento', 'z_reg_Lluvia', 'z_reg_Radiacion_Solar', 'z_reg_DURATION MINUTES', 'z_reg_EFFORT DISTANCE KM']

Cuartiles usados para grupos de Shannon:
Q1 = 1.499
Q3 = 2.543


,grupo,n
0,Media_Q2_Q3,16653
1,Alta_Q3,8328
2,Baja_Q1,8327


## 5. Tratamiento de valores atipicos

Los atipicos se marcaron con el criterio `|z| > 3`, pero **no se eliminaron automaticamente**. La razon es sustantiva: en contaminacion atmosferica, lluvia, viento, radiacion o esfuerzo de muestreo, los extremos pueden representar episodios reales, por ejemplo picos de contaminacion, condiciones meteorologicas particulares o jornadas de observacion largas. Eliminarlos sin una verificacion externa podria borrar precisamente eventos ambientales relevantes para la biodiversidad.

Por tanto, el tratamiento aplicado fue:

1. Detectar y documentar atipicos mediante variables `outlier_*`.
2. Conservar las observaciones para los analisis principales.
3. Usar transformaciones logaritmicas y estandarizacion cuando el metodo lo requiere.
4. Interpretar resultados con cautela cuando las distribuciones se alejan de normalidad.

In [6]:
outlier_counts = (
    pd.Series(prep_summary["outlier_counts_abs_z_gt_3"], name="n_atipicos")
    .rename_axis("variable_z")
    .reset_index()
)
outlier_counts["pct_analysis"] = (outlier_counts["n_atipicos"] / len(analysis) * 100).round(2)
display(outlier_counts.sort_values("n_atipicos", ascending=False))

outlier_cols = [c for c in analysis.columns if c.startswith("outlier_z_env_")]
analysis_outlier_profile = analysis.assign(n_atipicos_env=analysis[outlier_cols].sum(axis=1))
display(
    analysis_outlier_profile
    .sort_values("n_atipicos_env", ascending=False)
    [["Fecha_Hora", "Estacion", "Shannon_Index", "PM2.5", "PM10", "NO2", "CO", "O3", "Temperatura", "Humedad", "Viento", "Precipitacion", "Radiacion_Solar", "n_atipicos_env"]]
    .head(10)
)

,variable_z,n_atipicos,pct_analysis
3,z_env_CO,107,0.32
4,z_env_O3,100,0.30
14,z_reg_O3,83,0.25
21,z_reg_EFFORT DISTANCE KM,83,0.25
2,z_env_NO2,78,0.23
7,z_env_Viento,77,0.23
0,z_env_PM2.5,73,0.22
13,z_reg_CO,73,0.22
12,z_reg_NO2,69,0.21
17,z_reg_Viento,66,0.20


,Fecha_Hora,Estacion,Shannon_Index,PM2.5,PM10,NO2,CO,O3,Temperatura,Humedad,Viento,Precipitacion,Radiacion_Solar,n_atipicos_env
4033,2021-03-06 08:00:00,KENNEDY,2.593172,54.0,107.9,42.1,2.53,7.6,11.2,79.9,0.3,0.0,107.0,4
23145,2024-10-11 09:00:00,KENNEDY,2.398201,47.9,94.2,54.0,1.98,11.7,14.6,68.8,1.1,0.0,323.0,4
13795,2023-03-31 09:00:00,CENTRO DE ALTO RENDIMIENTO,2.162296,61.0,87.0,41.1,1.55,9.7,16.9,66.0,0.8,0.0,305.0,3
12144,2022-11-01 09:00:00,TUNAL,2.653357,48.0,87.0,37.4,2.22,12.7,18.2,56.0,0.7,0.0,437.0,3
13617,2023-02-24 11:00:00,JAZMIN,1.920675,50.7,86.9,44.0,0.99,46.4,18.8,44.0,1.6,0.0,752.0,3
13807,2023-03-31 09:00:00,CENTRO DE ALTO RENDIMIENTO,2.511574,61.0,87.0,41.1,1.55,9.7,16.9,66.0,0.8,0.0,305.0,3
12123,2022-11-02 09:00:00,JAZMIN,2.484715,44.8,88.9,44.6,2.91,0.5,14.7,83.0,0.9,0.0,122.0,3
13613,2023-02-24 11:00:00,JAZMIN,1.976630,50.7,86.9,44.0,0.99,46.4,18.8,44.0,1.6,0.0,752.0,3
13923,2023-03-08 13:00:00,SAN CRISTOBAL,1.767009,89.0,89.0,49.5,1.40,4.1,16.7,63.0,0.8,0.0,93.0,3
12706,2022-12-12 13:00:00,PUENTE ARANDA,1.673118,58.8,111.3,45.5,0.72,16.1,19.7,47.0,1.5,0.0,663.0,3


## 6. Diccionario minimo del dataset analitico

La siguiente tabla resume las variables centrales que se usan en los notebooks posteriores.

In [7]:
dictionary = pd.DataFrame([
    {"variable": "SAMPLING EVENT IDENTIFIER", "rol": "ID", "descripcion": "Identificador unico de la lista eBird"},
    {"variable": "Fecha_Hora", "rol": "llave temporal", "descripcion": "Hora de inicio de observacion truncada/cruzada con RMCAB"},
    {"variable": "Estacion", "rol": "llave espacial", "descripcion": "Estacion RMCAB mas cercana"},
    {"variable": "Distance_to_Station_km", "rol": "control espacial", "descripcion": "Distancia entre lista eBird y estacion asignada"},
    {"variable": "Shannon_Index", "rol": "respuesta", "descripcion": "Indice de diversidad de Shannon por lista"},
    {"variable": "Riqueza_Especies", "rol": "ecologica", "descripcion": "Numero de especies registradas en la lista"},
    {"variable": "PM2.5, PM10, NO2, CO, O3", "rol": "predictoras", "descripcion": "Contaminantes atmosfericos horarios"},
    {"variable": "Temperatura, Humedad, Viento, Precipitacion, Radiacion_Solar", "rol": "predictoras", "descripcion": "Variables meteorologicas horarias"},
    {"variable": "DURATION MINUTES, EFFORT DISTANCE KM", "rol": "controles", "descripcion": "Esfuerzo de muestreo de eBird"},
    {"variable": "Diversidad_Grupo", "rol": "grupo", "descripcion": "Baja_Q1, Media_Q2_Q3 o Alta_Q3 segun Shannon"},
])
display(dictionary)

,variable,rol,descripcion
0,SAMPLING EVENT IDENTIFIER,ID,Identificador unico de la lista eBird
1,Fecha_Hora,llave temporal,Hora de inicio de observacion truncada/cruzada...
2,Estacion,llave espacial,Estacion RMCAB mas cercana
3,Distance_to_Station_km,control espacial,Distancia entre lista eBird y estacion asignada
4,Shannon_Index,respuesta,Indice de diversidad de Shannon por lista
5,Riqueza_Especies,ecologica,Numero de especies registradas en la lista
6,"PM2.5, PM10, NO2, CO, O3",predictoras,Contaminantes atmosfericos horarios
7,"Temperatura, Humedad, Viento, Precipitacion, R...",predictoras,Variables meteorologicas horarias
8,"DURATION MINUTES, EFFORT DISTANCE KM",controles,Esfuerzo de muestreo de eBird
9,Diversidad_Grupo,grupo,"Baja_Q1, Media_Q2_Q3 o Alta_Q3 segun Shannon"


## 7. Chequeos finales de calidad

Antes de usar el dataset en EDA, distribuciones conjuntas, inferencia multivariada, regresion y reduccion de dimensionalidad, se verifican aspectos basicos de consistencia: identificadores duplicados, rango espacial del cruce, cobertura temporal y presencia de columnas clave.

In [8]:
required_columns = [
    "SAMPLING EVENT IDENTIFIER", "Fecha_Hora", "Estacion", "Distance_to_Station_km",
    "Shannon_Index", "PM2.5", "PM10", "NO2", "CO", "O3", "Temperatura", "Humedad",
    "Viento", "Lluvia", "Radiacion_Solar", "Diversidad_Grupo"
]

quality_checks = pd.DataFrame([
    {"chequeo": "Sin IDs duplicados en master", "resultado": master["SAMPLING EVENT IDENTIFIER"].duplicated().sum() == 0},
    {"chequeo": "Sin IDs duplicados en analysis", "resultado": analysis["SAMPLING EVENT IDENTIFIER"].duplicated().sum() == 0},
    {"chequeo": "Distancia maxima <= 5 km", "resultado": master["Distance_to_Station_km"].max() <= 5},
    {"chequeo": "Shannon sin nulos", "resultado": analysis["Shannon_Index"].notna().all()},
    {"chequeo": "Columnas clave presentes", "resultado": set(required_columns).issubset(analysis.columns)},
    {"chequeo": "Grupos de diversidad asignados", "resultado": analysis["Diversidad_Grupo"].notna().all()},
])
display(quality_checks)

station_counts = (
    master["Estacion"]
    .value_counts()
    .rename_axis("Estacion")
    .reset_index(name="listas_asignadas")
)
display(station_counts)

,chequeo,resultado
0,Sin IDs duplicados en master,True
1,Sin IDs duplicados en analysis,True
2,Distancia maxima <= 5 km,True
3,Shannon sin nulos,True
4,Columnas clave presentes,True
5,Grupos de diversidad asignados,True


,Estacion,listas_asignadas
0,CENTRO DE ALTO RENDIMIENTO,5266
1,FONTIBON,4543
2,MINAMBIENTE,3938
3,LAS FERIAS,2704
4,SUBA,2317
5,MOVIL 7MA,2053
6,BOLIVIA,1896
7,USAQUEN,1637
8,GUAYMARAL,1460
9,COLINA,1406


## 8. Dataset recomendado para cada etapa

- `analysis_dataset.parquet`: auditoria general, conteos, revision de nulos y exploraciones que no requieren casos completos.
- `analysis_env_complete.parquet`: EDA ambiental, distribuciones conjuntas, inferencia multivariada y PCA/FA cuando se necesitan las 10 variables ambientales.
- `analysis_regression_complete.parquet`: regresion con variables ambientales y controles de esfuerzo.
- `analysis_env_reduced_no_radiation.parquet` y `analysis_regression_reduced_no_radiation.parquet`: analisis de sensibilidad cuando se desea recuperar tamano muestral excluyendo `Radiacion_Solar`.

En sintesis, la limpieza elimina nulos para los datasets finales de modelamiento, corrige valores fisicamente imposibles, conserva atipicos plausibles y deja variables derivadas estandarizadas para hacer comparables las escalas ambientales.